# ETL: Bronze -> Silver

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
df = spark.table("dev_workspace.bronze.raw_spotify_tracks")
display(df)

In [0]:
print(f"Total rows in Bronze : {df.count():,}")
print(f"Schema:")
df.printSchema()

In [0]:
df_typed = (
    df
    .select(
        col("track_id"),
        col("track_name"),
        col("artists"),                        # kept as STRING for now, parsed in next cell
        col("album_name"),
        col("track_genre"),
        col("popularity").cast(IntegerType()),
        col("duration_ms").cast(LongType()),
        col("explicit").cast(BooleanType()),
        col("danceability").cast(DoubleType()),
        col("energy").cast(DoubleType()),
        col("key").cast(IntegerType()),        # integer first, mapped to label next
        col("loudness").cast(DoubleType()),
        col("mode").cast(IntegerType()),       # integer first, mapped to label next
        col("speechiness").cast(DoubleType()),
        col("acousticness").cast(DoubleType()),
        col("instrumentalness").cast(DoubleType()),
        col("liveness").cast(DoubleType()),
        col("valence").cast(DoubleType()),
        col("tempo").cast(DoubleType()),
        col("time_signature").cast(IntegerType()),
        col("_ingested_at"),
        col("_source_file"),
    )
)

print(f"Schema after casting:")
df_typed.printSchema()

display(df_typed)

In [0]:
key_mapping = create_map(
    lit(0),  lit("C"),
    lit(1),  lit("C#"),
    lit(2),  lit("D"),
    lit(3),  lit("D#"),
    lit(4),  lit("E"),
    lit(5),  lit("F"),
    lit(6),  lit("F#"),
    lit(7),  lit("G"),
    lit(8),  lit("G#"),
    lit(9),  lit("A"),
    lit(10), lit("A#"),
    lit(11), lit("B"),
)

df_transformed = (
    df_typed
    # Split comma-separated artists string into ARRAY<STRING> and trim each element
    .withColumn(
        "artists",
        transform(split(col("artists"), ";"), lambda x: trim(x))
    )
    # Map key integer → musical note label; null if key value is outside 0–11
    .withColumn("key",  key_mapping[col("key")])
    # Map mode integer → Major / Minor
    .withColumn("mode", when(col("mode") == 1, "Major")
                         .when(col("mode") == 0, "Minor")
                         .otherwise(None))
    # Normalize genre: trim whitespace + lowercase
    .withColumn("track_genre", lower(trim(col("track_genre"))))
    # Trim track and album names
    .withColumn("track_name", trim(col("track_name")))
    .withColumn("album_name", trim(col("album_name")))
)


display(df_transformed.select("track_id", "track_name", "artists", "key", "mode", "track_genre").limit(10))

In [0]:
df_clean.createOrReplaceTempView("df_clean")

result = spark.sql("""
    select *
    from df_clean
    where duration_ms is null or duration_ms > 0
""")
display(result)

In [0]:
total_before = df_transformed.count()

df_clean = (
    df_transformed
    .filter(col("track_id").isNotNull())
    .filter(col("track_name").isNotNull())
    #.filter(col("popularity").isNull()  | col("popularity").between(0, 100))
    #.filter(col("duration_ms").isNull() | (col("duration_ms") > 0))
    #.filter(col("tempo").isNull()       | (col("tempo") > 0))
)

total_after = df_clean.count()
dropped     = total_before - total_after

print(f"Rows before filtering : {total_before:,}")
print(f"Rows after filtering  : {total_after:,}")
print(f"Rows dropped          : {dropped:,}")

In [0]:
window_spec = Window.partitionBy("track_id").orderBy(F.col("_ingested_at").desc())

df_deduped = (
    df_clean
    .withColumn("_row_num", row_number().over(window_spec))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

total_clean  = df_clean.count()
total_deduped = df_deduped.count()

print(f"Rows before dedup : {total_clean:,}")
print(f"Rows after dedup  : {total_deduped:,}")
print(f"Duplicates removed: {total_clean - total_deduped:,}")

In [0]:
df = spark.sql("""
         SELECT *
FROM dev_workspace.bronze.raw_spotify_tracks
WHERE popularity > 90
          """)
display(df)
